# 01 · Data quality checks

**Goal:** make sure the combined *Company Movements* table is complete and internally consistent before any analysis.

Each row is one counterparty company for one consulting supplier: how many people moved **from** the supplier to that company (`departures`) and **to** the supplier from that company (`hires`) over the 12 months up to 2023-10-18.

| Column | Meaning |
|---|---|
| `supplier` | Anonymized consulting supplier (S01–S41) |
| `counterparty` | Anonymized company the people moved to/from (C0001…) |
| `is_consulting` | 1 if the counterparty is another consulting company |
| `is_automotive` | 1 if the counterparty is an automotive company |
| `departures` / `hires` | People who left to / joined from the counterparty |
| `net_change` | `hires − departures` |
| `ratio` | Signed hires/departures ratio (negative = the supplier loses more than it gains) |

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/company_movements.csv")
print(f"{df.shape[0]:,} rows · {df.shape[1]} columns · {df['supplier'].nunique()} suppliers")
df.head()

2,330 rows · 8 columns · 41 suppliers


,supplier,counterparty,is_consulting,is_automotive,departures,hires,net_change,ratio
0,S01,C0410,1,0,6,0,-6,-6.000000
1,S01,C1124,0,0,6,0,-6,-6.000000
2,S01,C1323,0,0,4,3,-1,-1.333333
3,S01,C0351,1,0,3,4,1,1.333333
4,S01,C0514,0,1,2,3,1,1.500000


In [2]:
df.dtypes

supplier             str
counterparty         str
is_consulting      int64
is_automotive      int64
departures         int64
hires              int64
net_change         int64
ratio            float64
dtype: object

## Missing values, duplicates and negative counts

In [3]:
checks = pd.Series({
    "missing values": int(df.isna().sum().sum()),
    "duplicated supplier–counterparty pairs": int(df.duplicated(["supplier", "counterparty"]).sum()),
    "negative departures": int((df["departures"] < 0).sum()),
    "negative hires": int((df["hires"] < 0).sum()),
    "flags not 0/1": int((~df[["is_consulting", "is_automotive"]].isin([0, 1])).sum().sum()),
})
checks

missing values                            0
duplicated supplier–counterparty pairs    0
negative departures                       0
negative hires                            0
flags not 0/1                             0
dtype: int64

## Are the derived columns consistent with the counts?

In [4]:
def signed_ratio(departures, hires):
    if departures == 0:
        return hires
    if hires == 0:
        return -departures
    if departures > hires:
        return -departures / hires
    return hires / departures

net_ok = (df["net_change"] == df["hires"] - df["departures"]).mean()
ratio_ok = np.isclose(df["ratio"], [signed_ratio(d, h) for d, h in zip(df["departures"], df["hires"])]).mean()
print(f"net_change consistent: {net_ok:.0%}")
print(f"ratio consistent:      {ratio_ok:.0%}")

net_change consistent: 100%
ratio consistent:      100%


> **Fixed while reviewing the project (2026):** for the three suppliers whose subsidiaries were merged into one report, the original merge step calculated the ratio as *departures / hires* instead of *hires / departures* when hires were higher (8 of 2,330 rows). The definition is corrected in `src/prepare_data.py`, and the ratio is recalculated from the counts in `src/anonymize.py`, so all suppliers now use the same definition.

## Distribution of the counts

In [5]:
df[["departures", "hires", "net_change", "ratio"]].describe().round(2)

,departures,hires,net_change,ratio
count,2330.00,2330.00,2330.00,2330.00
mean,0.71,0.92,0.22,0.27
std,1.48,1.13,1.71,1.71
min,0.00,0.00,-25.00,-25.00
25%,0.00,0.00,-1.00,-1.00
50%,0.00,1.00,1.00,1.00
75%,1.00,1.00,1.00,1.00
max,25.00,19.00,11.00,11.00


In [6]:
rows_per_supplier = df.groupby("supplier").size()
print(f"Counterparties per supplier: min {rows_per_supplier.min()}, median {rows_per_supplier.median():.0f}, max {rows_per_supplier.max()}")
print(f"Share of rows with 0 departures: {(df['departures'] == 0).mean():.0%}")
print(f"Share of rows with 0 hires:      {(df['hires'] == 0).mean():.0%}")

Counterparties per supplier: min 4, median 44, max 209
Share of rows with 0 departures: 59%
Share of rows with 0 hires:      30%


## Summary

- No missing values, no duplicated supplier–counterparty pairs, no negative counts, and the category flags only contain 0/1.
- `net_change` matches `hires − departures` for every row, and `ratio` now follows one definition for all suppliers.
- The counts are small and skewed (most counterparties exchange 0–2 people), which matters for the choice of statistical test in notebook 03.